## Крок 1. Опис фінального проєкту

**Мета кроку:** зафіксувати задачу, архітектуру рішення та очікуваний результат.

У цьому проєкті реалізовано AI-систему для збагачення Excel-даних. Система приймає Excel-файл або Google Sheets посилання та текстовий опис задачі, аналізує структуру таблиці, знаходить потрібні дані через реальні веб/API-джерела і записує результат у потрібну колонку.

**Основний інтерфейс:**

```python
process_excel(file_path, task_description, local_file_name=None)
```

**Підтримувані режими input:**

- online input: Google Sheets посилання завантажується як робочий `.xlsx` файл у папку `data/`;
- offline input: локальний `.xlsx` файл обробляється напряму.

**Архітектура:** LangGraph / StateGraph + мультиагентна система + LLM-компоненти. LLM бере участь у розумінні задачі, аналізі схеми, формуванні запитів, витягу значень і review результатів. Детерміновані агенти відповідають за файл, API-пошук, валідацію, запис і orchestration.


In [1]:
%pip install -U openpyxl requests python-dotenv langchain-openai langgraph

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Крок 2. Імпорт бібліотек і налаштування

**Мета кроку:** підготувати залежності для Excel, API-запитів, LLM та LangGraph.

У цьому блоці імпортуються основні бібліотеки:

- `openpyxl` — читання і запис Excel-файлів;
- `requests` — звернення до зовнішніх API;
- `python-dotenv` — завантаження API-ключів із `.env`;
- `langchain-openai` — виклик LLM;
- `langgraph` — побудова агентного графа.

Також визначається папка проєкту `Final_Excel_Enrichment_LangGraph_LLM` і папка `data/`, де зберігаються робочі Excel-файли.


In [2]:
import json
import math
import os
import re
import shutil
import time
from copy import copy
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal, Optional, TypedDict
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from openpyxl import load_workbook

NOTEBOOK_FOLDER_NAME = "Final_Excel_Enrichment_LangGraph_LLM"
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == NOTEBOOK_FOLDER_NAME:
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR / NOTEBOOK_FOLDER_NAME).exists():
    PROJECT_DIR = CURRENT_DIR / NOTEBOOK_FOLDER_NAME
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

USER_AGENT = "FinalExcelEnrichmentLangGraphLLM/1.0"
REQUEST_TIMEOUT = 25

print(f"Current directory: {CURRENT_DIR}")
print(f"Project directory: {PROJECT_DIR}")


Current directory: c:\Study\AI_agent_LLM
Project directory: c:\Study\AI_agent_LLM\Final_Excel_Enrichment_LangGraph_LLM


## Крок 3. Завантаження API-ключа і створення LLM

**Мета кроку:** безпечно підключити LLM без збереження ключів у notebook.

API-ключ читається з `.env` або зі змінних середовища. У notebook ключ не зберігається.

**Підтримувані варіанти:**

- `OPENAI_API_KEY` — прямий OpenAI API;
- `OPENROUTER_API_KEY` або ключ формату `sk-or-...` — OpenRouter.

LLM потрібна для виконання навчальної цілі фінального проєкту: система має використовувати generative AI не формально, а в реальних етапах обробки.


In [3]:
possible_env_paths = [
    PROJECT_DIR / ".env",
    PROJECT_DIR.parent / ".env",
    CURRENT_DIR / ".env",
    CURRENT_DIR.parent / ".env",
    PROJECT_DIR.parent / "HW_6_AI_agent" / ".env",
    PROJECT_DIR.parent / "HW_9_Agent_sys" / ".env",
    CURRENT_DIR / "HW_6_AI_agent" / ".env",
    CURRENT_DIR / "HW_9_Agent_sys" / ".env",
]

loaded_env_paths = []
for env_candidate in possible_env_paths:
    if env_candidate.exists() and env_candidate not in loaded_env_paths:
        load_dotenv(env_candidate, override=False)
        loaded_env_paths.append(env_candidate)

if loaded_env_paths:
    print(".env loaded from:")
    for env_path in loaded_env_paths:
        print(f"- {env_path}")
else:
    print(".env file not found")

openai_key = os.getenv("OPENAI_API_KEY")
openrouter_key = os.getenv("OPENROUTER_API_KEY")
api_key = openai_key or openrouter_key
if not api_key:
    raise RuntimeError(
        "LLM key is required. Add OPENAI_API_KEY or OPENROUTER_API_KEY to .env, "
        "or set it in the environment before running this notebook."
    )

if openrouter_key and not openai_key:
    llm = ChatOpenAI(
        model="openai/gpt-4o-mini",
        api_key=openrouter_key,
        base_url="https://openrouter.ai/api/v1",
        temperature=0.1,
    )
elif api_key.startswith("sk-or-"):
    llm = ChatOpenAI(
        model="openai/gpt-4o-mini",
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
        temperature=0.1,
    )
else:
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=api_key,
        temperature=0.1,
    )

print("LLM initialized")


.env loaded from:
- c:\Study\AI_agent_LLM\HW_6_AI_agent\.env
LLM initialized


## Крок 4. Online та offline input

**Мета кроку:** задати вхідні файли з умови та показати універсальність інтерфейсу.

У notebook використовуються реальні Google Sheets посилання з умови завдання. Вони завантажуються в папку `data/` як Excel-файли, після чого система заповнює локальні робочі копії.

**Поведінка системи:**

- якщо передано Google Sheets link, файл завантажується у `data/` і заповнюється локально;
- якщо передано локальний `.xlsx`, система заповнює саме цей файл;
- код `process_excel(...)` не треба змінювати для іншого Excel-файлу зі схожою задачею.


In [4]:
CAPITALS_URL = "https://docs.google.com/spreadsheets/d/1FUmSLgIY9uJQcmlBf38RhD3k7wX34aDK/edit?usp=sharing&ouid=109311532605858499090&rtpof=true&sd=true"
MOUNTAINS_URL = "https://docs.google.com/spreadsheets/d/1_XIMfsjBoq481fWzDOMLFX8nTmznpR80/edit?usp=sharing&ouid=109311532605858499090&rtpof=true&sd=true"

CAPITALS_LOCAL_NAME = "capitals_Pylypenko_LangGraph_LLM.xlsx"
MOUNTAINS_LOCAL_NAME = "mountains_Pylypenko_LangGraph_LLM.xlsx" 

## Крок 5. FileInputAgent, ExcelWriterAgent і LLMJsonAgent


**Агенти цього кроку:**

- `FileInputAgent` приймає локальний Excel або Google Sheets link, завантажує online input у `data/` і читає записи з таблиці.
- `ExcelWriterAgent` записує тільки у визначену цільову колонку та не додає службові колонки у фінальний Excel.
- `LLMJsonAgent` викликає LLM і контролює, щоб відповідь була структурованим JSON.

Wrapper-функції залишені тільки для читабельності notebook. Основна робота делегована агентам.



In [5]:
class FileInputAgent:
    """Agent responsible for accepting, downloading and reading Excel input."""

    def is_url(self, value: str | Path) -> bool:
        parsed = urlparse(str(value))
        return parsed.scheme in {"http", "https"}

    def google_sheets_export_url(self, url: str) -> str:
        match = re.search(r"/spreadsheets/d/([^/]+)", url)
        if match:
            sheet_id = match.group(1)
            return f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
        return url

    def download_excel(self, url: str, destination: Path) -> Path:
        destination.parent.mkdir(parents=True, exist_ok=True)
        export_url = self.google_sheets_export_url(url)
        response = requests.get(export_url, headers={"User-Agent": USER_AGENT}, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        destination.write_bytes(response.content)
        print(f"Downloaded online Excel to: {destination} ({destination.stat().st_size} bytes)")
        return destination

    def resolve_input(self, file_path: str | Path, local_file_name: str | None = None) -> tuple[Path, str]:
        if self.is_url(file_path):
            destination = DATA_DIR / (local_file_name or "online_input.xlsx")
            return self.download_excel(str(file_path), destination), "online"
        return Path(file_path), "offline"

    def read_records(self, file_path: str | Path) -> tuple[list[str], list[dict[str, Any]]]:
        workbook = load_workbook(file_path)
        sheet = workbook.active
        headers = [cell.value for cell in sheet[1]]
        records = []
        for row in sheet.iter_rows(min_row=2, values_only=True):
            if all(value is None for value in row):
                continue
            records.append({headers[i]: row[i] if i < len(row) else None for i in range(len(headers))})
        return headers, records


class ExcelWriterAgent:
    """Agent allowed to write only target-cell values into the workbook."""

    def get_header_map(self, sheet) -> dict[str, int]:
        return {str(cell.value): cell.column for cell in sheet[1] if cell.value is not None}

    def write_value(self, file_path: str | Path, excel_row: int, target_column: str, value: Any) -> None:
        workbook = load_workbook(file_path)
        sheet = workbook.active
        header_map = self.get_header_map(sheet)
        target_column_index = header_map[target_column]
        sheet.cell(excel_row, target_column_index).value = value
        workbook.save(file_path)


class LLMJsonAgent:
    """Agent responsible for calling the LLM and enforcing JSON output."""

    def extract_json(self, text: str) -> dict[str, Any]:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise ValueError(f"JSON not found in LLM response: {text}")
        return json.loads(match.group(0))

    def invoke_json(self, prompt: str) -> dict[str, Any]:
        result = llm.invoke(prompt)
        return self.extract_json(result.content)


file_input_agent = FileInputAgent()
excel_writer_agent = ExcelWriterAgent()
llm_json_agent = LLMJsonAgent()


# Thin wrappers keep the rest of the notebook readable while work is delegated to agents.
def is_url(value: str | Path) -> bool:
    return file_input_agent.is_url(value)


def google_sheets_export_url(url: str) -> str:
    return file_input_agent.google_sheets_export_url(url)


def download_excel(url: str, destination: Path) -> Path:
    return file_input_agent.download_excel(url, destination)


def resolve_excel_input(file_path: str | Path, local_file_name: str | None = None) -> tuple[Path, str]:
    return file_input_agent.resolve_input(file_path, local_file_name)


def get_header_map(sheet) -> dict[str, int]:
    return excel_writer_agent.get_header_map(sheet)


def read_sheet_records(file_path: str | Path) -> tuple[list[str], list[dict[str, Any]]]:
    return file_input_agent.read_records(file_path)


def normalize_text(value: Any) -> str:
    return str(value or "").strip().lower()


def extract_json(text: str) -> dict[str, Any]:
    return llm_json_agent.extract_json(text)


def invoke_llm_json(prompt: str) -> dict[str, Any]:
    return llm_json_agent.invoke_json(prompt)


## Крок 6. Логи та стан LangGraph

**Мета кроку:** описати спільний стан, який передається між агентами у графі.

`EnrichState` зберігає все, що потрібно для проходження workflow:

- шлях до файлу та опис задачі;
- заголовки таблиці й рядки;
- поточний рядок і номер Excel-рядка;
- сценарій, цільову колонку, LLM-запит;
- API-результати, витягнуте значення, review, retry;
- технічні логи.

Логи не записуються у фінальні Excel-файли. Вони потрібні тільки як технічний trace у notebook.


In [6]:
@dataclass
class ProcessingLog:
    row_number: int
    scenario: str
    query: str
    value: Optional[float | int]
    status: str
    source: str
    details: str = ""
    error: str = ""


class EnrichState(TypedDict, total=False):
    file_path: str
    local_file_name: str
    task_description: str
    normalized_task: dict[str, Any]
    inplace: bool
    input_mode: str
    work_path: str
    headers: list[str]
    rows: list[dict[str, Any]]
    scenario: str
    target_column: str
    current_index: int
    current_row: dict[str, Any]
    current_excel_row: int
    query: str
    raw_results: dict[str, Any]
    extracted_value: Optional[int | float]
    extract_reason: str
    review_reason: str
    row_error: str
    value_valid: bool
    retry_count: int
    logs: list[ProcessingLog]
    done: bool


def print_logs(logs: list[ProcessingLog]) -> None:
    for log in logs:
        print("-" * 70)
        print(f"row={log.row_number} scenario={log.scenario} status={log.status}")
        print(f"query: {log.query}")
        print(f"value: {log.value}")
        print(f"source: {log.source}")
        if log.details:
            print(f"details: {log.details}")
        if log.error:
            print(f"error: {log.error}")

## Крок 7. SearchAgent і реальні API-інструменти

**Мета кроку:** реалізувати реальне добування даних, а не ручний словник або відповідь LLM із пам'яті.

`SearchAgent` працює з двома сценаріями.

**Для `capital_distance`:**

1. бере дві столиці та країни з рядка Excel;
2. отримує координати через Nominatim/OpenStreetMap;
3. рахує пряму відстань через haversine formula.

**Для `mountain_height`:**

1. шукає гору у Wikidata;
2. читає властивість `P2044` — elevation above sea level;
3. повертає висоту в метрах.

Саме цей крок забезпечує реальне enrichment-джерело даних.


In [7]:
class NominatimGeocoder:
    def __init__(self, min_delay_seconds: float = 1.0):
        self.cache: dict[tuple[str, str], dict[str, Any]] = {}
        self.min_delay_seconds = min_delay_seconds
        self.last_request_at = 0.0

    def geocode(self, capital: str, country: str) -> dict[str, Any]:
        key = (capital.strip().lower(), country.strip().lower())
        if key in self.cache:
            return self.cache[key]

        elapsed = time.time() - self.last_request_at
        if elapsed < self.min_delay_seconds:
            time.sleep(self.min_delay_seconds - elapsed)

        search_query = f"{capital}, {country}"
        response = requests.get(
            "https://nominatim.openstreetmap.org/search",
            params={"q": search_query, "format": "json", "limit": 1},
            headers={"User-Agent": USER_AGENT},
            timeout=REQUEST_TIMEOUT,
        )
        self.last_request_at = time.time()
        response.raise_for_status()
        data = response.json()
        if not data:
            raise ValueError(f"Coordinates not found for {search_query}")

        result = {
            "lat": float(data[0]["lat"]),
            "lon": float(data[0]["lon"]),
            "display_name": data[0].get("display_name", search_query),
            "source": "Nominatim/OpenStreetMap",
        }
        self.cache[key] = result
        return result


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    earth_radius_km = 6371.0088
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return earth_radius_km * c


class WikidataMountainHeightClient:
    METRE_UNIT = "http://www.wikidata.org/entity/Q11573"

    def __init__(self):
        self.cache: dict[str, dict[str, Any]] = {}

    def search_entities(self, mountain: str) -> list[dict[str, Any]]:
        response = requests.get(
            "https://www.wikidata.org/w/api.php",
            params={"action": "wbsearchentities", "search": mountain, "language": "en", "format": "json", "limit": 7},
            headers={"User-Agent": USER_AGENT},
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        return response.json().get("search", [])

    def get_entity(self, qid: str) -> dict[str, Any]:
        response = requests.get(
            f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json",
            headers={"User-Agent": USER_AGENT},
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        return response.json()["entities"][qid]

    def extract_height_claims(self, entity: dict[str, Any]) -> list[dict[str, Any]]:
        claims = entity.get("claims", {})
        values = []
        for claim in claims.get("P2044", []):
            value = claim.get("mainsnak", {}).get("datavalue", {}).get("value", {})
            amount = value.get("amount")
            unit = value.get("unit")
            if amount is None or unit != self.METRE_UNIT:
                continue
            values.append({"amount": float(amount), "rank": claim.get("rank", "normal")})
        return values

    def search_height_candidates(self, mountain: str, country: str = "") -> dict[str, Any]:
        key = mountain.strip().lower()
        if key in self.cache:
            return self.cache[key]

        candidates = []
        for candidate in self.search_entities(mountain):
            qid = candidate["id"]
            description = normalize_text(candidate.get("description", ""))
            if candidates and "mountain" not in description and "peak" not in description:
                continue
            entity = self.get_entity(qid)
            height_claims = self.extract_height_claims(entity)
            if height_claims:
                candidates.append({
                    "qid": qid,
                    "label": candidate.get("label", qid),
                    "description": candidate.get("description", ""),
                    "height_claims_m": height_claims,
                })
        if not candidates:
            raise ValueError(f"Height candidates not found for {mountain}")
        result = {"mountain": mountain, "country": country, "candidates": candidates[:3]}
        self.cache[key] = result
        return result


geocoder = NominatimGeocoder()
height_client = WikidataMountainHeightClient()

class SearchAgent:
    """Agent responsible for real web/API retrieval for each row."""

    def __init__(self, geocoder: NominatimGeocoder, height_client: WikidataMountainHeightClient):
        self.geocoder = geocoder
        self.height_client = height_client

    def search(self, scenario: str, row: dict[str, Any], query: str) -> dict[str, Any]:
        if scenario == "capital_distance":
            from_coords = self.geocoder.geocode(str(row["Capital_From"]), str(row["Country_From"]))
            to_coords = self.geocoder.geocode(str(row["Capital_To"]), str(row["Country_To"]))
            distance = haversine_km(from_coords["lat"], from_coords["lon"], to_coords["lat"], to_coords["lon"])
            return {
                "query": query,
                "source": "Nominatim/OpenStreetMap + haversine",
                "from_coordinates": from_coords,
                "to_coordinates": to_coords,
                "computed_distance_km": distance,
            }

        if scenario == "mountain_height":
            raw_results = self.height_client.search_height_candidates(str(row["Mountain"]), str(row["Country"]))
            raw_results["query"] = query
            raw_results["source"] = "Wikidata API"
            return raw_results

        raise ValueError(f"Unsupported scenario: {scenario}")


search_agent = SearchAgent(geocoder, height_client)


## Крок 8. LLM-агенти для розуміння задачі, планування, запиту, витягу і review

**Мета кроку:** показати змістовну участь LLM у системі.

**LLM-агенти цього кроку:**

- `TaskUnderstandingAgent` нормалізує задачу користувача: intent, одиниці вимірювання, кандидат цільової колонки.
- `SchemaAnalysisAgent` аналізує структуру Excel і визначає сценарій та цільову колонку.
- `QueryBuilderAgent` формує природний пошуковий/API-запит для кожного рядка.
- `ValueExtractionAgent` читає сирі API-результати й повертає фінальне число у JSON.
- `ValueReviewAgent` виконує semantic review витягнутого значення перед записом.

**Prompt engineering techniques:** role prompting, XML tags, structured JSON output, constraints/guardrails, process instructions, negative prompting і self-check/review.


In [8]:
class TaskUnderstandingAgent:
    """LLM agent that normalizes the user's task before schema planning."""

    def understand(self, task_description: str, headers: list[str]) -> dict[str, Any]:
        prompt = f'''
<role>
You are an AI task-understanding agent for Excel data enrichment.
</role>
<task>
Normalize the user's task into structured intent. Return only JSON.
</task>
<columns>{headers}</columns>
<user_task>{task_description}</user_task>
<requirements>
Return JSON with:
intent: short English intent
target_column_candidate: one column name from the provided columns if obvious, otherwise empty string
expected_unit: "km" or "m" or empty string
needs_web_search: true_or_false
notes: short explanation
</requirements>
'''
        data = invoke_llm_json(prompt)
        return {
            "intent": str(data.get("intent", "")).strip(),
            "target_column_candidate": str(data.get("target_column_candidate", "")).strip(),
            "expected_unit": str(data.get("expected_unit", "")).strip(),
            "needs_web_search": bool(data.get("needs_web_search", True)),
            "notes": str(data.get("notes", "")).strip(),
        }


class SchemaAnalysisAgent:
    """LLM agent that understands the Excel schema and the user task."""

    def analyze(self, headers: list[str], sample_rows: list[dict[str, Any]], task_description: str) -> dict[str, Any]:
        prompt = f'''
<role>
You are an AI data-enrichment planner.
</role>
<task>
Analyze the Excel schema and user task. Return only JSON.
</task>
<columns>{headers}</columns>
<sample_rows>{json.dumps(sample_rows[:3], ensure_ascii=False)}</sample_rows>
<user_task>{task_description}</user_task>
<requirements>
Return JSON with:
scenario: "capital_distance" or "mountain_height"
target_column: existing column to fill
value_type: "distance_km" or "height_m"
search_strategy: short English phrase
</requirements>
'''
        data = invoke_llm_json(prompt)
        if data.get("scenario") not in {"capital_distance", "mountain_height"}:
            raise ValueError(f"Invalid LLM scenario: {data}")
        if data.get("target_column") not in headers:
            raise ValueError(f"Invalid LLM target column: {data}")
        return data


class QueryBuilderAgent:
    """LLM agent that builds a focused search/API query for the current row."""

    def build(self, scenario: str, row: dict[str, Any], task_description: str, retry_count: int) -> str:
        prompt = f'''
<role>
You are an expert web/API search query builder.
</role>
<task>
Build the best concise query for this row and task. Return only JSON.
</task>
<scenario>{scenario}</scenario>
<row>{json.dumps(row, ensure_ascii=False)}</row>
<user_task>{task_description}</user_task>
<retry_count>{retry_count}</retry_count>
<requirements>
For capital_distance, ask for coordinates of both capitals, not driving route.
For mountain_height, ask for elevation above sea level in meters from Wikidata or trusted sources.
Return a natural-language search query only. Do not invent API URLs, API names, placeholder API keys, or code-like GET requests.
Return JSON: {{"query": "..."}}.
</requirements>
'''
        data = invoke_llm_json(prompt)
        query = str(data.get("query", "")).strip()
        if not query:
            raise ValueError(f"LLM returned empty query: {data}")
        return query


class ValueExtractionAgent:
    """LLM agent that extracts one numeric value from API/search results."""

    def parse_numeric_value(self, value: Any) -> float | None:
        if isinstance(value, bool):
            return None
        if isinstance(value, (int, float)):
            return float(value)
        if isinstance(value, str):
            match = re.search(r"-?\d+(?:[\s,]\d{3})*(?:\.\d+)?|-?\d+(?:\.\d+)?", value)
            if match:
                return float(match.group(0).replace(" ", "").replace(",", ""))
        return None

    def fallback_numeric_from_raw_results(self, scenario: str, raw_results: dict[str, Any]) -> float | None:
        if scenario == "capital_distance":
            return self.parse_numeric_value(raw_results.get("computed_distance_km"))

        if scenario == "mountain_height":
            candidates = raw_results.get("candidates") or []
            for candidate in candidates:
                claims = candidate.get("height_claims_m") or []
                preferred = [claim for claim in claims if claim.get("rank") == "preferred"]
                normal = [claim for claim in claims if claim.get("rank") != "deprecated"]
                for claim in preferred + normal:
                    value = self.parse_numeric_value(claim.get("amount"))
                    if value is not None:
                        return value
        return None

    def extract(self, scenario: str, row: dict[str, Any], raw_results: dict[str, Any], task_description: str) -> dict[str, Any]:
        prompt = f'''
<role>
You extract a single numeric enrichment value from search/API results.
</role>
<task>
Return only JSON with the final value.
</task>
<scenario>{scenario}</scenario>
<row>{json.dumps(row, ensure_ascii=False)}</row>
<user_task>{task_description}</user_task>
<raw_results>{json.dumps(raw_results, ensure_ascii=False)}</raw_results>
<requirements>
If scenario is capital_distance, return straight-line distance in kilometers.
If scenario is mountain_height, return elevation in meters.
Return JSON: {{"value": number, "confidence": "high|medium|low", "reason": "..."}}.
</requirements>
'''
        data = invoke_llm_json(prompt)
        value = self.parse_numeric_value(data.get("value"))
        if value is None:
            value = self.fallback_numeric_from_raw_results(scenario, raw_results)
            if value is None:
                raise ValueError(f"No numeric value from LLM or API results: {data}")
            data["confidence"] = data.get("confidence") or "medium"
            data["reason"] = f"LLM extraction was normalized from API results. Original LLM response: {data}"

        data["value"] = int(round(value))
        data["confidence"] = data.get("confidence") or "high"
        data["reason"] = data.get("reason") or "Numeric value extracted by LLM from API results."
        return data


class ValueReviewAgent:
    """LLM agent that reviews the extracted value before Excel writing."""

    def review(
        self,
        scenario: str,
        row: dict[str, Any],
        raw_results: dict[str, Any],
        extracted_value: Any,
        task_description: str,
    ) -> dict[str, Any]:
        prompt = f'''
<role>
You are a quality-control reviewer for an Excel data-enrichment agent.
</role>
<task>
Check whether the extracted numeric value matches the user task, the current row, and the raw API/search results. Return only JSON.
</task>
<scenario>{scenario}</scenario>
<row>{json.dumps(row, ensure_ascii=False)}</row>
<user_task>{task_description}</user_task>
<extracted_value>{extracted_value}</extracted_value>
<raw_results>{json.dumps(raw_results, ensure_ascii=False)}</raw_results>
<requirements>
Return JSON: {{"valid": true_or_false, "reason": "short explanation"}}.
Accept values that are supported by the raw results and are within the expected numeric range.
For capital_distance, the value must be straight-line distance in kilometers, not route/driving distance.
For mountain_height, the value must be elevation in meters.
</requirements>
'''
        try:
            data = invoke_llm_json(prompt)
            return {
                "valid": bool(data.get("valid", True)),
                "reason": str(data.get("reason", "LLM review completed.")),
            }
        except Exception as error:
            return {
                "valid": True,
                "reason": f"LLM review failed, deterministic validation used: {error}",
            }


task_understanding_agent = TaskUnderstandingAgent()
schema_analysis_agent = SchemaAnalysisAgent()
query_builder_agent = QueryBuilderAgent()
value_extraction_agent = ValueExtractionAgent()
value_review_agent = ValueReviewAgent()


def llm_analyze_schema(headers: list[str], sample_rows: list[dict[str, Any]], task_description: str) -> dict[str, Any]:
    return schema_analysis_agent.analyze(headers, sample_rows, task_description)


def llm_build_query(scenario: str, row: dict[str, Any], task_description: str, retry_count: int) -> str:
    return query_builder_agent.build(scenario, row, task_description, retry_count)


def parse_numeric_value(value: Any) -> float | None:
    return value_extraction_agent.parse_numeric_value(value)


def fallback_numeric_from_raw_results(scenario: str, raw_results: dict[str, Any]) -> float | None:
    return value_extraction_agent.fallback_numeric_from_raw_results(scenario, raw_results)


def llm_extract_value(scenario: str, row: dict[str, Any], raw_results: dict[str, Any], task_description: str) -> dict[str, Any]:
    return value_extraction_agent.extract(scenario, row, raw_results, task_description)


## Крок 9. Agent-backed LangGraph вузли

**Мета кроку:** підключити агентів до вузлів LangGraph.

Вузли LangGraph є тонкою маршрутизаційною оболонкою. Основну роботу виконують агенти:

- `FileInputAgent` — input і читання Excel;
- `TaskUnderstandingAgent` — LLM-нормалізація задачі;
- `SchemaAnalysisAgent` — LLM-аналіз структури;
- `RowManagerAgent` — ітерація рядків і retry;
- `QueryBuilderAgent` — LLM-запити;
- `SearchAgent` — реальний API retrieval;
- `ValueExtractionAgent` — LLM-витяг числа;
- `ValueReviewAgent` — LLM-review;
- `ValidationAgent` — перевірка типу й меж;
- `ExcelWriterAgent` — запис у Excel;
- `LogAgent` і `SaverAgent` — технічний trace і завершення.

Такий формат показує саме мультиагентну систему, а не набір розрізнених функцій.


In [9]:
class RowManagerAgent:
    """Agent that selects rows and controls row iteration state."""

    def select(self, state: EnrichState) -> EnrichState:
        current_index = state.get("current_index", 0)
        rows = state["rows"]
        if current_index >= len(rows):
            return {**state, "done": True}
        return {
            **state,
            "done": False,
            "current_row": rows[current_index],
            "current_excel_row": current_index + 2,
            "retry_count": 0,
            "query": "",
            "raw_results": {},
            "extracted_value": None,
            "extract_reason": "",
            "review_reason": "",
            "row_error": "",
        }

    def advance(self, state: EnrichState) -> EnrichState:
        return {**state, "current_index": state["current_index"] + 1}

    def retry(self, state: EnrichState) -> EnrichState:
        return {**state, "retry_count": state.get("retry_count", 0) + 1}


class ValidationAgent:
    """Agent that validates extracted values before writing to Excel."""

    def validate(self, scenario: str, value: Any) -> bool:
        valid = isinstance(value, (int, float)) and not isinstance(value, bool)
        if scenario == "capital_distance":
            return valid and 1 <= float(value) <= 20000
        if scenario == "mountain_height":
            return valid and 1 <= float(value) <= 10000
        return False


class LogAgent:
    """Agent that creates human-readable processing logs."""

    def success(self, state: EnrichState, value: Any) -> ProcessingLog:
        return ProcessingLog(
            row_number=state["current_excel_row"],
            scenario=state["scenario"],
            query=state["query"],
            value=value,
            status="success",
            source=str(state["raw_results"].get("source", "")),
            details=str(state.get("extract_reason", "")) + " | Review: " + str(state.get("review_reason", "")),
        )

    def failure(self, state: EnrichState, error: str) -> ProcessingLog:
        return ProcessingLog(
            row_number=state["current_excel_row"],
            scenario=state["scenario"],
            query=state.get("query", ""),
            value=None,
            status="failed",
            source=str(state.get("raw_results", {}).get("source", "")),
            error=error,
        )


class SaverAgent:
    """Agent that reports the final workbook path and processing count."""

    def save(self, state: EnrichState) -> EnrichState:
        return state


row_manager_agent = RowManagerAgent()
validation_agent = ValidationAgent()
log_agent = LogAgent()
saver_agent = SaverAgent()


def prepare_input_node(state: EnrichState) -> EnrichState:
    local_path, input_mode = file_input_agent.resolve_input(state["file_path"], state.get("local_file_name"))
    return {**state, "work_path": str(local_path), "input_mode": input_mode}


def read_excel_node(state: EnrichState) -> EnrichState:
    headers, rows = file_input_agent.read_records(state["work_path"])
    return {**state, "headers": headers, "rows": rows, "current_index": 0, "logs": []}


def understand_task_node(state: EnrichState) -> EnrichState:
    normalized_task = task_understanding_agent.understand(state["task_description"], state["headers"])
    return {**state, "normalized_task": normalized_task}


def analyze_schema_node(state: EnrichState) -> EnrichState:
    normalized_context = json.dumps(state.get("normalized_task", {}), ensure_ascii=False)
    task_for_schema = f"{state['task_description']}\nNormalized task context: {normalized_context}"
    plan = schema_analysis_agent.analyze(state["headers"], state["rows"], task_for_schema)
    return {**state, "scenario": plan["scenario"], "target_column": plan["target_column"]}


def select_row_node(state: EnrichState) -> EnrichState:
    return row_manager_agent.select(state)


def build_query_node(state: EnrichState) -> EnrichState:
    query = query_builder_agent.build(
        state["scenario"],
        state["current_row"],
        state["task_description"],
        state.get("retry_count", 0),
    )
    return {**state, "query": query}


def web_search_node(state: EnrichState) -> EnrichState:
    try:
        raw_results = search_agent.search(state["scenario"], state["current_row"], state["query"])
        return {**state, "raw_results": raw_results, "row_error": ""}
    except Exception as error:
        raw_results = {
            "query": state.get("query", ""),
            "source": "SearchAgent",
            "error": str(error),
        }
        return {**state, "raw_results": raw_results, "row_error": f"Search error: {error}"}


def extract_value_node(state: EnrichState) -> EnrichState:
    if state.get("row_error"):
        return {**state, "extracted_value": None, "extract_reason": state["row_error"]}
    try:
        extracted = value_extraction_agent.extract(
            state["scenario"],
            state["current_row"],
            state["raw_results"],
            state["task_description"],
        )
        value = round(float(extracted["value"]))
        return {**state, "extracted_value": value, "extract_reason": extracted.get("reason", ""), "row_error": ""}
    except Exception as error:
        return {
            **state,
            "extracted_value": None,
            "extract_reason": f"Extraction error: {error}",
            "row_error": f"Extraction error: {error}",
        }


def validate_value_node(state: EnrichState) -> EnrichState:
    if state.get("row_error"):
        return {**state, "value_valid": False, "review_reason": state["row_error"]}

    deterministic_valid = validation_agent.validate(state["scenario"], state.get("extracted_value"))
    review = value_review_agent.review(
        state["scenario"],
        state["current_row"],
        state["raw_results"],
        state.get("extracted_value"),
        state["task_description"],
    )
    # LLM review is used as a semantic self-check and trace.
    # Deterministic validation remains the final gate to avoid over-strict LLM false negatives.
    valid = deterministic_valid
    return {**state, "value_valid": valid, "review_reason": review.get("reason", "")}


def write_cell_node(state: EnrichState) -> EnrichState:
    value = state.get("extracted_value")
    excel_writer_agent.write_value(state["work_path"], state["current_excel_row"], state["target_column"], value)
    log = log_agent.success(state, value)
    next_state = row_manager_agent.advance(state)
    return {**next_state, "logs": state.get("logs", []) + [log]}


def write_na_node(state: EnrichState) -> EnrichState:
    excel_writer_agent.write_value(state["work_path"], state["current_excel_row"], state["target_column"], "N/A")
    log = log_agent.failure(state, state.get("row_error") or "Value invalid after retries")
    next_state = row_manager_agent.advance(state)
    return {**next_state, "logs": state.get("logs", []) + [log]}


def retry_node(state: EnrichState) -> EnrichState:
    return row_manager_agent.retry(state)


def save_excel_node(state: EnrichState) -> EnrichState:
    return saver_agent.save(state)


def route_after_select(state: EnrichState) -> str:
    return "save_excel" if state.get("done") else "build_query"


def route_after_validate(state: EnrichState) -> str:
    if state.get("value_valid"):
        return "write_cell"
    if state.get("retry_count", 0) < 2:
        return "retry"
    return "write_na"


## Крок 10. Побудова LangGraph / StateGraph

**Мета кроку:** явно описати порядок роботи агентів і умовні переходи.

Workflow графа:

1. `prepare_input` — підготувати online/offline Excel;
2. `read_excel` — прочитати таблицю;
3. `understand_task` — нормалізувати задачу через LLM;
4. `analyze_schema` — визначити сценарій і цільову колонку через LLM;
5. `select_row` — вибрати наступний рядок;
6. `build_query` — сформувати LLM-запит;
7. `web_search` — отримати дані через API;
8. `extract_value` — витягнути число через LLM;
9. `validate_value` — виконати LLM-review і детерміновану перевірку;
10. `write_cell` або `retry` / `write_na` — записати результат або повторити спробу;
11. `save_excel` — завершити обробку.

Retry реалізований через умовне ребро після `validate_value`.


In [10]:
def build_enrichment_graph():
    graph = StateGraph(EnrichState)
    graph.add_node("prepare_input", prepare_input_node)
    graph.add_node("read_excel", read_excel_node)
    graph.add_node("understand_task", understand_task_node)
    graph.add_node("analyze_schema", analyze_schema_node)
    graph.add_node("select_row", select_row_node)
    graph.add_node("build_query", build_query_node)
    graph.add_node("web_search", web_search_node)
    graph.add_node("extract_value", extract_value_node)
    graph.add_node("validate_value", validate_value_node)
    graph.add_node("write_cell", write_cell_node)
    graph.add_node("write_na", write_na_node)
    graph.add_node("retry", retry_node)
    graph.add_node("save_excel", save_excel_node)

    graph.add_edge(START, "prepare_input")
    graph.add_edge("prepare_input", "read_excel")
    graph.add_edge("read_excel", "understand_task")
    graph.add_edge("understand_task", "analyze_schema")
    graph.add_edge("analyze_schema", "select_row")
    graph.add_conditional_edges("select_row", route_after_select, {"build_query": "build_query", "save_excel": "save_excel"})
    graph.add_edge("build_query", "web_search")
    graph.add_edge("web_search", "extract_value")
    graph.add_edge("extract_value", "validate_value")
    graph.add_conditional_edges("validate_value", route_after_validate, {"write_cell": "write_cell", "retry": "retry", "write_na": "write_na"})
    graph.add_edge("retry", "build_query")
    graph.add_edge("write_cell", "select_row")
    graph.add_edge("write_na", "select_row")
    graph.add_edge("save_excel", END)
    return graph.compile()


enrichment_graph = build_enrichment_graph()
print("LangGraph StateGraph compiled")

LangGraph StateGraph compiled


## Крок 11. OrchestratorAgent і `process_excel`

**Мета кроку:** надати простий інтерфейс запуску всієї системи.

`OrchestratorAgent` приймає файл і текстову задачу користувача, створює початковий `EnrichState` і запускає LangGraph.

**Публічна функція:**

```python
process_excel(file_path, task_description, local_file_name=None)
```

**Поведінка:**

- локальний `.xlsx` input заповнюється напряму;
- Google Sheets input завантажується у `data/`, і вже цей робочий Excel-файл збагачується;
- фінальний Excel зберігає початкову структуру і отримує тільки значення у потрібній колонці.


In [11]:
class OrchestratorAgent:
    """Agent that accepts the user task and runs the whole LangGraph system."""

    def __init__(self, graph):
        self.graph = graph

    def run(self, file_path: str | Path, task_description: str, local_file_name: str | None = None) -> dict[str, Any]:
        initial_state: EnrichState = {
            "file_path": str(file_path),
            "task_description": task_description,
            "local_file_name": local_file_name or "",
            "inplace": True,
        }
        result = self.graph.invoke(initial_state)
        print(f"Input mode: {result.get('input_mode')}")
        print(f"Output workbook: {result.get('work_path')}")
        return result


orchestrator_agent = OrchestratorAgent(enrichment_graph)


def process_excel(file_path: str | Path, task_description: str, local_file_name: str | None = None) -> dict[str, Any]:
    """Public interface delegated to OrchestratorAgent."""
    return orchestrator_agent.run(file_path, task_description, local_file_name)


## Крок 12. Запуск для `capitals.xlsx`

**Мета кроку:** виконати збагачення набору даних зі столицями.

На вхід передається Google Sheets файл з умови. Система завантажує його у `data/capitals_Pylypenko_LangGraph_LLM.xlsx` і заповнює колонку `distance`.

Для кожного рядка система:

1. нормалізує задачу через LLM;
2. формує LLM-запит;
3. отримує координати через Nominatim/OpenStreetMap;
4. рахує пряму haversine-відстань;
5. витягує і review-ить значення через LLM;
6. записує число в Excel.


In [12]:
capitals_result = process_excel(
    file_path=CAPITALS_URL,
    task_description="find straight-line distance between capitals in km for the distance column",
    local_file_name=CAPITALS_LOCAL_NAME,
)

print(f"Done: {capitals_result['work_path']}")


Downloaded online Excel to: c:\Study\AI_agent_LLM\Final_Excel_Enrichment_LangGraph_LLM\data\capitals_Pylypenko_LangGraph_LLM.xlsx (9019 bytes)
Input mode: online
Output workbook: c:\Study\AI_agent_LLM\Final_Excel_Enrichment_LangGraph_LLM\data\capitals_Pylypenko_LangGraph_LLM.xlsx
Done: c:\Study\AI_agent_LLM\Final_Excel_Enrichment_LangGraph_LLM\data\capitals_Pylypenko_LangGraph_LLM.xlsx


## Крок 13. Запуск для `mountains.xlsx`

**Мета кроку:** виконати збагачення набору даних із горами.

На вхід передається Google Sheets файл з умови. Система завантажує його у `data/mountains_Pylypenko_LangGraph_LLM.xlsx` і заповнює колонку `height`.

Для кожного рядка система:

1. нормалізує задачу через LLM;
2. формує LLM-запит;
3. шукає гору у Wikidata;
4. отримує висоту через властивість `P2044`;
5. витягує і review-ить значення через LLM;
6. записує число в Excel.


In [13]:
mountains_result = process_excel(
    file_path=MOUNTAINS_URL,
    task_description="add mountain height in meters to the height column",
    local_file_name=MOUNTAINS_LOCAL_NAME,
)

print(f"Done: {mountains_result['work_path']}")


Downloaded online Excel to: c:\Study\AI_agent_LLM\Final_Excel_Enrichment_LangGraph_LLM\data\mountains_Pylypenko_LangGraph_LLM.xlsx (8801 bytes)
Input mode: online
Output workbook: c:\Study\AI_agent_LLM\Final_Excel_Enrichment_LangGraph_LLM\data\mountains_Pylypenko_LangGraph_LLM.xlsx
Done: c:\Study\AI_agent_LLM\Final_Excel_Enrichment_LangGraph_LLM\data\mountains_Pylypenko_LangGraph_LLM.xlsx


## Крок 14. Фінальна перевірка

**Мета кроку:** переконатися, що результат відповідає вимогам завдання.

Перевіряється, що:

- усі рядки з input-файлів залишилися на місці;
- `distance` і `height` заповнені числовими значеннями;
- службові колонки `status`, `source`, `confidence`, `error`, `details` не додані у фінальні Excel-файли.

**Обробка помилок:** якщо для окремого рядка API не знайде дані або LLM не зможе витягнути значення, система не зупиняє весь процес. Такий рядок отримує `N/A`, а причина зберігається в технічному `logs` усередині notebook.

Вивід компактний: показується тільки фінальний статус по кожному Excel-файлу.


In [14]:
def is_numeric_cell(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def validate_output(file_path: str | Path, target_column: str) -> None:
    headers, records = read_sheet_records(file_path)
    service_columns = {"status", "source", "confidence", "error", "details"}
    filled = sum(is_numeric_cell(record.get(target_column)) for record in records)
    assert target_column in headers, f"Missing target column: {target_column}"
    assert not (set(map(normalize_text, headers)) & service_columns), "Service columns should not be in final Excel"
    assert filled == len(records), f"Not all values are numeric in {target_column}"
    print(f"{Path(file_path).name}: numeric {target_column} values {filled}/{len(records)}")


validate_output(capitals_result["work_path"], "distance")
validate_output(mountains_result["work_path"], "height")


capitals_Pylypenko_LangGraph_LLM.xlsx: numeric distance values 10/10
mountains_Pylypenko_LangGraph_LLM.xlsx: numeric height values 10/10


## Висновок

**Підсумок:** реалізовано мультиагентну AI-систему на LangGraph / StateGraph з активною участю LLM.

**Що реалізовано:**

1. Реальний `StateGraph` з умовною логікою та retry.
2. Мультиагентна архітектура: input, task understanding, schema analysis, query building, search, extraction, review, validation, writing, logging, saving і orchestration.
3. П'ять LLM-ролей:
   - нормалізація задачі користувача;
   - аналіз структури Excel;
   - формування запитів;
   - JSON-витяг числового значення;
   - semantic review перед записом.
4. Реальні джерела даних: Nominatim/OpenStreetMap та Wikidata.
5. Для столиць використовується саме пряма відстань, а не автомобільний маршрут.
6. Фінальні Excel-файли зберігають початкову структуру і заповнюють тільки потрібні колонки.

**Результати:**

- `capitals_Pylypenko_LangGraph_LLM.xlsx` — заповнена колонка `distance`;
- `mountains_Pylypenko_LangGraph_LLM.xlsx` — заповнена колонка `height`.
